# 1. Libraries

In [1]:
# This notebook will be doing the preprocessing of the data for the model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from imblearn.over_sampling import SMOTE


# 2. Data load

In [2]:
# Lets import the data
path = '../Data/Titanic-Dataset.csv'
df = pd.read_csv(path)
# Lets check the data
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 3. Preprocessing

In [3]:
# First, we will frop the columns that are not needed for the model
temp_df = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']]

# Then, let's separate the dataasets. One for men the other one for women
male_df = temp_df.loc[temp_df.Sex == "male"]
female_df = temp_df.loc[temp_df.Sex == "female"]

In [4]:
# Pipeline to impute only the 'age' column
imputer_pipeline = Pipeline(steps=[
    ('imputer', IterativeImputer(random_state=42))
])

# Impute 'Age' column for males
male_df.loc[:, 'Age'] = imputer_pipeline.fit_transform(male_df.loc[:, ['Age']])

# Impute 'Age' column for females
female_df.loc[:, 'Age'] = imputer_pipeline.fit_transform(female_df.loc[:, ['Age']])

# Step 2: Scale Fare into a new column 'Fare_scaled'

# Scale 'Fare' for males
male_df.loc[:, 'Fare_scaled'] = np.sqrt(male_df.loc[:, 'Fare'])  # Adding 1 to avoid log(0)

# Scale 'Fare' for females
female_df.loc[:, 'Fare_scaled'] = np.sqrt(female_df.loc[:, 'Fare'])

# Dropping the 'Sex' column in both dataframes
male_df = male_df.drop('Sex', axis=1)
female_df = female_df.drop('Sex', axis=1)


C:\Users\sebas\AppData\Local\Temp\ipykernel_24636\4013907710.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  male_df.loc[:, 'Fare_scaled'] = np.sqrt(male_df.loc[:, 'Fare'])  # Adding 1 to avoid log(0)
C:\Users\sebas\AppData\Local\Temp\ipykernel_24636\4013907710.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  female_df.loc[:, 'Fare_scaled'] = np.sqrt(female_df.loc[:, 'Fare'])


In [5]:
male_df.Survived.value_counts()

Survived
0    468
1    109
Name: count, dtype: int64

In [6]:
female_df.Survived.value_counts()

Survived
1    233
0     81
Name: count, dtype: int64

The rest of the variables are either counts or categories. So for this model, we will use a RandomForestClassifier which is able to handle categories and counts with ease.

In [7]:
# Finally lets split the data into train and test sets

#For male

X_train_male, X_test_male, y_train_male, y_test_male = train_test_split(male_df.drop(columns=['Survived']), 
                                                                    male_df['Survived'],
                                                                    stratify= male_df['Survived'],
                                                                    test_size=0.15, 
                                                                    random_state=42)    

X_train_fem, X_test_fem, y_train_fem, y_test_fem = train_test_split(female_df.drop(columns=['Survived']), 
                                                                    female_df['Survived'], 
                                                                    stratify=female_df['Survived'],
                                                                    test_size=0.15, 
                                                                    random_state=42)    

# Using SMOTE to balance the dataset
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled_male = smote.fit_resample(X_train_male, y_train_male)
X_train_resampled_fem, y_train_resampled_fem = smote.fit_resample(X_train_fem, y_train_fem)

In [8]:
# Export them to csv

train_path = r'../Data/Train'
test_path = r'../Data/Test'


# X datasets
X_train_resampled.to_csv(train_path + '/Xmale_train.csv', index=False)
X_train_resampled_fem.to_csv(train_path + '/Xfemale_train.csv', index=False)
X_test_male.to_csv(test_path + '/Xmale_test.csv', index=False)
X_test_fem.to_csv(test_path + '/Xfemale_test.csv', index=False)  

# y datasets
y_train_resampled_male.to_csv(train_path + '/ymale_train.csv', index=False)
y_train_resampled_fem.to_csv(train_path + '/yfemale_train.csv', index=False)
y_test_male.to_csv(test_path + '/ymale_test.csv', index=False)
y_test_fem.to_csv(test_path + '/yfemale_test.csv', index=False)

